**BPE tokenizer from scratch**

Build a working Byte-Pair Encoding tokenizer to understand how GPT-style tokenization works

In [4]:
from collections import Counter


class SimpleBPETokenizer:
    # A minimal BPE tokenizer that shows how pairs get merged

    def __init__(self):
        self.vocab = {}          # Token string -> integer ID
        self.reverse_vocab = {}  # Integer ID -> token string
        self.merges = []         # Learned merge rules in order

    def get_pairs(self, tokens):
        # Count all adjacent token pairs in a word
        pairs = Counter()

        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i + 1])] += 1

        return pairs

    def train(self, text, vocab_size=50):
        # Split corpus into individual words
        words = text.split()

        # Count how many times each word appears
        raw_word_freqs = Counter(words)

        # Represent each word as a sequence of individual characters
        # and add the end-of-word token
        word_freqs = {
            tuple(list(word) + ['</w>']): freq
            for word, freq in raw_word_freqs.items()
        }

        # Extract all unique characters and the end-of-word token
        all_chars = set()

        for tokens in word_freqs.keys():
            all_chars.update(tokens)

        # Create the initial vocabulary
        self.vocab = {
            token: i
            for i, token in enumerate(sorted(all_chars))
        }

        # Create reverse mapping from ID to token
        self.reverse_vocab = {
            i: token
            for token, i in self.vocab.items()
        }

        # Keep merging pairs until the target vocabulary size is reached
        while len(self.vocab) < vocab_size:

            # Count adjacent pairs across the entire corpus
            pair_counts = Counter()

            for tokens, freq in word_freqs.items():
                pairs = self.get_pairs(tokens)

                # Multiply pair frequency by word frequency
                for pair, count in pairs.items():
                    pair_counts[pair] += count * freq

            # Stop if there are no pairs left to merge
            if not pair_counts:
                break

            # Select the most frequent pair in the corpus
            best_pair = pair_counts.most_common(1)[0][0]

            # Combine the pair to create a new token
            new_token = best_pair[0] + best_pair[1]

            # Assign the next available vocabulary ID
            new_id = len(self.vocab)

            self.vocab[new_token] = new_id
            self.reverse_vocab[new_id] = new_token

            # Store the merge rule so it can be reused during tokenization
            self.merges.append((best_pair, new_token))

            # Apply the selected merge to every word
            new_word_freqs = {}

            for tokens, freq in word_freqs.items():
                tokens = list(tokens)
                merged_tokens = []
                i = 0

                # Scan the token sequence and merge matching adjacent pairs
                while i < len(tokens):
                    if (
                        i < len(tokens) - 1
                        and (tokens[i], tokens[i + 1]) == best_pair
                    ):
                        # Replace the pair with the new merged token
                        merged_tokens.append(new_token)

                        # Skip both tokens that were merged
                        i += 2
                    else:
                        # Keep the current token unchanged
                        merged_tokens.append(tokens[i])
                        i += 1

                # Store the updated token sequence
                new_word_freqs[tuple(merged_tokens)] = freq

            # Use the newly merged words for the next iteration
            word_freqs = new_word_freqs

    def tokenize(self, text):
        words = text.split()
        all_tokens = []

        for word in words:
            # Start with individual characters and the end-of-word token
            tokens = list(word) + ['</w>']

            # Apply learned merge rules in the same order they were learned
            for pair, new_token in self.merges:
                i = 0

                # Find and merge every occurrence of the current pair
                while i < len(tokens) - 1:
                    if (tokens[i], tokens[i + 1]) == pair:
                        tokens = (
                            tokens[:i]
                            + [new_token]
                            + tokens[i + 2:]
                        )
                    else:
                        i += 1

            all_tokens.extend(tokens)

        return all_tokens

    def encode(self, text):
        # Convert text into tokens and then map tokens to IDs
        tokens = self.tokenize(text)

        return [
            self.vocab.get(token, 0)
            for token in tokens
        ]

    def decode(self, ids):
        # Convert IDs back into token strings
        tokens = [
            self.reverse_vocab.get(i, '?')
            for i in ids
        ]

        # Join tokens and replace end-of-word markers with spaces
        text = ''.join(tokens).replace('</w>', ' ')

        return text.strip()


corpus = """
the cat sat on the mat
the dog sat on the log
the cat and the dog played
"""

tokenizer = SimpleBPETokenizer()

tokenizer.train(
    corpus,
    vocab_size=30
)

for text in [
    'the cat',
    'the dog sat',
    'cat and dog'
]:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text)

    print(f"\nText: '{text}'")
    print(f"  Tokens: {tokens}")
    print(f"  IDs:    {ids}")
    print(f"  Decoded: '{tokenizer.decode(ids)}'")


Text: 'the cat'
  Tokens: ['the</w>', 'cat</w>']
  IDs:    [17, 22]
  Decoded: 'the cat'

Text: 'the dog sat'
  Tokens: ['the</w>', 'dog</w>', 'sat</w>']
  IDs:    [17, 26, 23]
  Decoded: 'the dog sat'

Text: 'cat and dog'
  Tokens: ['cat</w>', 'a', 'n', 'd</w>', 'dog</w>']
  IDs:    [22, 1, 9, 27, 26]
  Decoded: 'cat and dog'
